<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/Attention-Based%20Image%20Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Attention-Based Image Classification
Implementing a Convolutional Neural Network (CNN) with an integrated Self-Attention mechanism.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

### Custom Attention Layer
This layer calculates attention weights to highlight important spatial features in the feature map.

In [2]:
class SpatialAttention(layers.Layer):
    def __init__(self):
        super(SpatialAttention, self).__init__()
        self.conv = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')

    def call(self, inputs):
        # Calculate average and max pooling across channels
        avg_pool = tf.reduce_mean(inputs, axis=-1, keepdims=True)
        max_pool = tf.reduce_max(inputs, axis=-1, keepdims=True)
        combined = tf.concat([avg_pool, max_pool], axis=-1)

        # Generate attention map
        attention = self.conv(combined)
        return inputs * attention

### Build CNN with Attention

In [3]:
def create_attention_cnn(input_shape=(64, 64, 3), num_classes=10):
    inputs = layers.Input(shape=input_shape)

    # Standard Conv Blocks
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)

    # Apply Attention Mechanism
    x = SpatialAttention()(x)

    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

model = create_attention_cnn()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_attention               │ (None, 32, 32, 64)     │            99 │
│ (SpatialAttention)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     1,048,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,068,781 (4.08 MB)

 Trainable params: 1,068,781 (4.08 MB)

 Non-trainable params: 0 (0.00 B)

### Train the Model
We will use synthetic data to demonstrate the training loop.

In [5]:
# Generate dummy data
x_train = np.random.random((100, 64, 64, 3))
y_train = np.random.randint(10, size=(100, 1))

# Train the model
print('Starting training...')
history = model.fit(x_train, y_train, epochs=5, batch_size=16)
print('Training complete.')

Starting training...
Epoch 1/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.1100 - loss: 2.2988
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - accuracy: 0.1300 - loss: 2.2918
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 106ms/step - accuracy: 0.1400 - loss: 2.2750
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - accuracy: 0.1800 - loss: 2.2528
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 107ms/step - accuracy: 0.1300 - loss: 2.2472
Training complete.


### Train the Model
We will use dummy data to demonstrate that the model compiles and trains correctly.

In [4]:
# Generate dummy data
x_train = np.random.random((100, 64, 64, 3))
y_train = np.random.randint(10, size=(100, 1))

# Train the model
print('Starting training...')
history = model.fit(x_train, y_train, epochs=5, batch_size=16)
print('Training complete.')

Starting training...
Epoch 1/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 111ms/step - accuracy: 0.0900 - loss: 2.4031
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.0900 - loss: 2.2974
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 108ms/step - accuracy: 0.1800 - loss: 2.2960
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 139ms/step - accuracy: 0.1800 - loss: 2.2874
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 190ms/step - accuracy: 0.1800 - loss: 2.2706
Training complete.
